# 🔧 02 — Feature Engineering

**Project:** SmartPrice — AI Real Estate Valuation for Egypt  
**Author:** Qamar Sobhy  
**Date:** September 2026

---

## 🎯 Objective

Transform raw listings into a **model-ready dataset** with engineered features:

1. Clean missing values & outliers
2. Engineer new features (ratios, logs)
3. Encode categorical variables
4. Extract NLP features
5. Save processed dataset for training

**Input:** `data/real_data/model_ready_clean.csv` (7,749 rows × 79 cols)  
**Output:** `data/processed/features.csv` (ready for model)


In [1]:
# ═══════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

print(f"✅ pandas: {pd.__version__}")


✅ pandas: 2.2.3


## 📂 1. Load Raw Data


In [2]:
# ═══════════════════════════════════════════════════════
# LOAD
# ═══════════════════════════════════════════════════════
DATA_PATH = Path('/content/drive/MyDrive/Smart-House-Price-Predictor_FINAL/data/real_data/model_ready_clean.csv')

df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# نبني نسخة للـ features
features = df.copy()


✅ Loaded: 7,749 rows × 79 cols


## 🧹 2. Data Cleaning

Remove outliers and handle invalid values.


In [3]:
# ═══════════════════════════════════════════════════════
# DATA CLEANING
# ═══════════════════════════════════════════════════════
print("Before cleaning:")
print(f"   Rows: {len(features):,}")

# 1) Remove rows without price or size
before = len(features)
features = features.dropna(subset=['price', 'size'])
print(f"\n1️⃣ Dropped missing price/size: {before - len(features)} rows")

# 2) Remove invalid values
before = len(features)
features = features[features['price'] > 0]
features = features[features['size'] >= 20]  # Realistic minimum
features = features[features['size'] <= 2000]  # Realistic maximum
print(f"2️⃣ Dropped invalid price/size: {before - len(features)} rows")

# 3) Remove price outliers (> 99.5th percentile)
before = len(features)
price_99 = features['price'].quantile(0.995)
features = features[features['price'] <= price_99]
print(f"3️⃣ Dropped price outliers (>{price_99/1e6:.1f}M): {before - len(features)} rows")

print(f"\n✅ Final size: {len(features):,} rows")
print(f"📊 Kept: {len(features) / len(df) * 100:.1f}% of original data")


Before cleaning:
   Rows: 7,749

1️⃣ Dropped missing price/size: 0 rows
2️⃣ Dropped invalid price/size: 0 rows
3️⃣ Dropped price outliers (>18.0M): 38 rows

✅ Final size: 7,711 rows
📊 Kept: 99.5% of original data


## ⚙️ 3. Feature Engineering

Create new features that help the model capture complex relationships.


In [4]:
# ═══════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════
print("🔧 Building engineered features...")

# 1) Bed/Bath ratio
features['bed_bath_ratio'] = (
    features['bedrooms'] / features['bathrooms'].replace(0, 1)
).round(2)
print("✅ bed_bath_ratio")

# 2) Area per bedroom
features['area_per_bedroom'] = (
    features['size'] / features['bedrooms'].replace(0, 1)
).round(1)
print("✅ area_per_bedroom")

# 3) Total rooms
features['rooms_total'] = features['bedrooms'] + features['bathrooms']
print("✅ rooms_total")

# 4) Log-transformed size (reduces skew)
features['log_size'] = np.log1p(features['size']).round(3)
print("✅ log_size")

# 5) Square root size
features['sqrt_size'] = np.sqrt(features['size']).round(2)
print("✅ sqrt_size")

# 6) Beds per m²
features['bed_per_sqm'] = (
    features['bedrooms'] / features['size']
).round(5)
print("✅ bed_per_sqm")

# 7) Baths per m²
features['bath_per_sqm'] = (
    features['bathrooms'] / features['size']
).round(5)
print("✅ bath_per_sqm")

# 8) Amenity count
amenity_cols = [c for c in features.columns if c.startswith('amenity_')]
if amenity_cols:
    features['amenity_count'] = features[amenity_cols].sum(axis=1)
    print(f"✅ amenity_count ({len(amenity_cols)} amenities)")

# 9) Price per m² (for analysis, not training)
features['price_per_sqm'] = (features['price'] / features['size']).round(0)
print("✅ price_per_sqm")

print(f"\n📊 Total features now: {features.shape[1]}")


🔧 Building engineered features...
✅ bed_bath_ratio
✅ area_per_bedroom
✅ rooms_total
✅ log_size
✅ sqrt_size
✅ bed_per_sqm
✅ bath_per_sqm
✅ amenity_count (33 amenities)
✅ price_per_sqm

📊 Total features now: 80


### 🗺️ Geographic Features

Distance to Cairo as a key location feature.


In [5]:
# ═══════════════════════════════════════════════════════
# GEOGRAPHIC FEATURES
# ═══════════════════════════════════════════════════════
from math import radians, cos, sin, asin, sqrt

# Cairo coordinates
CAIRO_LAT, CAIRO_LON = 30.0444, 31.2357

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance between two GPS points in km."""
    if pd.isna(lat1) or pd.isna(lon1):
        return np.nan
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Distance to Cairo
if 'latitude' in features.columns and 'longitude' in features.columns:
    features['distance_to_cairo'] = features.apply(
        lambda r: haversine(r['latitude'], r['longitude'], CAIRO_LAT, CAIRO_LON),
        axis=1
    ).round(2)
    print(f"✅ distance_to_cairo")
    print(f"   Mean: {features['distance_to_cairo'].mean():.1f} km")
    print(f"   Max: {features['distance_to_cairo'].max():.1f} km")

    # Geo cluster (simple grid-based)
    features['geo_cluster'] = (
        features['latitude'].round(1).astype(str) + '_' +
        features['longitude'].round(1).astype(str)
    )
    # Label encode
    features['geo_cluster'] = pd.factorize(features['geo_cluster'])[0]
    print(f"✅ geo_cluster: {features['geo_cluster'].nunique()} unique clusters")


✅ distance_to_cairo
   Mean: 68.3 km
   Max: 446.2 km
✅ geo_cluster: 80 unique clusters


### 🏷️ Categorical Encoding

Encode city, district, compound using One-Hot or Label encoding.


In [6]:
# ═══════════════════════════════════════════════════════
# CATEGORICAL ENCODING
# ═══════════════════════════════════════════════════════
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['city', 'district', 'compound']

for col in categorical_cols:
    if col in features.columns:
        # Fill missing with 'Unknown'
        features[col] = features[col].fillna('Unknown')
        features[col] = features[col].astype(str)

        # Label encode (numeric)
        le = LabelEncoder()
        features[f'{col}_encoded'] = le.fit_transform(features[col])
        print(f"✅ {col}: {features[col].nunique()} unique values → encoded")


✅ city: 9 unique values → encoded
✅ district: 43 unique values → encoded
✅ compound: 888 unique values → encoded


## 🎯 4. Feature Selection

Choose final features for the model.


In [7]:
# ═══════════════════════════════════════════════════════
# FEATURE SELECTION
# ═══════════════════════════════════════════════════════
# Target
target = 'price'

# Core numeric features
numeric_features = [
    'size', 'bedrooms', 'bathrooms',
    'log_size', 'sqrt_size',
    'bed_bath_ratio', 'area_per_bedroom', 'rooms_total',
    'bed_per_sqm', 'bath_per_sqm',
]

# Optional features
if 'amenity_count' in features.columns:
    numeric_features.append('amenity_count')
if 'distance_to_cairo' in features.columns:
    numeric_features.append('distance_to_cairo')
if 'geo_cluster' in features.columns:
    numeric_features.append('geo_cluster')
if 'title_length' in features.columns:
    numeric_features.append('title_length')
if 'title_word_count' in features.columns:
    numeric_features.append('title_word_count')

# Categorical encoded
for col in ['city', 'district', 'compound']:
    enc_col = f'{col}_encoded'
    if enc_col in features.columns:
        numeric_features.append(enc_col)

# Amenity binary features
amenity_features = [c for c in features.columns if c.startswith('amenity_')]

# NLP features
nlp_features = [c for c in features.columns if c.startswith('nlp_')]

# All features
all_features = numeric_features + amenity_features + nlp_features

# Remove duplicates
all_features = list(dict.fromkeys(all_features))

print(f"📊 Feature Selection:")
print(f"   Numeric:    {len(numeric_features)}")
print(f"   Amenities:  {len(amenity_features)}")
print(f"   NLP:        {len(nlp_features)}")
print(f"   ─────────────────")
print(f"   TOTAL:      {len(all_features)} features")
print()
print(f"🎯 Target: {target}")


📊 Feature Selection:
   Numeric:    18
   Amenities:  33
   NLP:        12
   ─────────────────
   TOTAL:      62 features

🎯 Target: price


## 💾 5. Save Processed Data

Save the final dataset for model training.


In [8]:
# ═══════════════════════════════════════════════════════
# SAVE PROCESSED DATA
# ═══════════════════════════════════════════════════════
# Create output folder
output_dir = Path('/content/drive/MyDrive/Smart-House-Price-Predictor_FINAL/data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare final dataset
final_df = features[all_features + [target]].copy()

# Drop rows with missing features
before = len(final_df)
final_df = final_df.dropna()
print(f"🧹 Dropped NaN rows: {before - len(final_df)}")

# Save
output_path = output_dir / 'features.csv'
final_df.to_csv(output_path, index=False)

print(f"\n✅ Saved: {output_path.name}")
print(f"📊 Shape: {final_df.shape[0]:,} rows × {final_df.shape[1]} cols")
print(f"📏 Size: {output_path.stat().st_size / 1024:.1f} KB")
print()
print("📋 Preview:")
display(final_df.head())


🧹 Dropped NaN rows: 0



✅ Saved: features.csv
📊 Shape: 7,711 rows × 63 cols
📏 Size: 1322.3 KB

📋 Preview:


,size,bedrooms,bathrooms,log_size,sqrt_size,bed_bath_ratio,area_per_bedroom,rooms_total,bed_per_sqm,bath_per_sqm,amenity_count,distance_to_cairo,geo_cluster,title_length,title_word_count,city_encoded,district_encoded,compound_encoded,amenity_balcony,amenity_builtin_wardrobes,amenity_covered_parking,amenity_private_garden,amenity_shared_pool,amenity_view,amenity_security,amenity_shared_spa,amenity_study,amenity_wc,amenity_balcony_2,amenity_lobby,amenity_children_area,amenity_air_conditioning,amenity_kitchen,amenity_maid_room,amenity_storage,amenity_dining_area,amenity_annex,amenity_cafeteria,amenity_gazebo,amenity_playground,amenity_gym,amenity_elevator,amenity_chiller,amenity_furnished,amenity_fully_furnished,amenity_garden,amenity_sport,amenity_shops,amenity_park,amenity_mosque,nlp_sea_view,nlp_garden,nlp_duplex,nlp_roof,nlp_furnished,nlp_new,nlp_luxury,nlp_open_view,nlp_parking,nlp_elevator,nlp_payment_plan,nlp_ready,price
0,158,2.0,3.0,5.069,12.57,0.67,79.0,5.0,0.01266,0.01899,26,28.99,0,48,9,3,39,702,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,5600000.0
1,155,3.0,3.0,5.050,12.45,1.00,51.7,6.0,0.01935,0.01935,4,35.15,1,50,9,2,28,558,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4400000.0
2,182,3.0,4.0,5.209,13.49,0.75,60.7,7.0,0.01648,0.02198,14,45.24,2,48,8,2,26,671,1,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6000000.0
3,125,2.0,2.0,4.836,11.18,1.00,62.5,4.0,0.01600,0.01600,12,45.24,2,43,6,2,26,671,1,0,0,0,0,1,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,4800000.0
4,135,2.0,3.0,4.913,11.62,0.67,67.5,5.0,0.01481,0.02222,10,22.49,3,46,8,3,39,620,1,0,0,0,0,0,0,0,0,1,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,9500000.0


## 🎯 6. Summary

### ✅ What We Did

1. **Cleaning**
   - Removed rows with missing price/size
   - Filtered unrealistic values
   - Cut price outliers at 99.5th percentile

2. **Engineered Features** (7 new)
   - `bed_bath_ratio`, `area_per_bedroom`, `rooms_total`
   - `log_size`, `sqrt_size`
   - `bed_per_sqm`, `bath_per_sqm`
   - `amenity_count`, `distance_to_cairo`, `geo_cluster`

3. **Categorical Encoding**
   - Label-encoded `city`, `district`, `compound`

4. **Selected Final Features**
   - Total: ~64 features
   - Target: `price`

### 📁 Output

**File:** `data/processed/features.csv`  
**Rows:** Model-ready dataset  
**Purpose:** Direct input to Notebook 3 (Model Training)

### 🎯 Next Steps

1. **03_model_training.ipynb** — Train XGBoost
2. **04_evaluation_shap.ipynb** — Evaluate & explain

---

### 📝 Conclusion

The dataset is now **model-ready** with engineered features, encoded categoricals, and clean values. This will feed directly into the training pipeline.
